## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack, csr_matrix
from scipy import stats

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from collections import Counter

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

RANDOM_STATE = 42
TARGET_COL = 'label'

print('Libraries imported.')

/kaggle/input/comment-category-prediction-challenge/Sample.csv
/kaggle/input/comment-category-prediction-challenge/train.csv
/kaggle/input/comment-category-prediction-challenge/test.csv
Libraries imported.


## 2. Load Data

In [2]:
BASE = '/kaggle/input/comment-category-prediction-challenge/'


# load train, test and sample submission files
train      = pd.read_csv(BASE + "train.csv")
test       = pd.read_csv(BASE + "test.csv")
sample_sub = pd.read_csv(BASE + "Sample.csv")

print("Train shape :", train.shape)
print("Test shape  :", test.shape)
print("Sample Sub  :", sample_sub.shape)

print("\nTrain Head:\n")
display(train.head())

print("\nTest Head:\n")
display(test.head())

# check column data types
print("\nData Types:\n")
display(train.dtypes)

# summary statistics for all columns
print("\nBasic Statistics:\n")
display(train.describe(include="all"))

Train shape : (198000, 15)
Test shape  : (102000, 14)
Sample Sub  : (102000, 2)

Train Head:



,created_date,post_id,emoticon_1,emoticon_2,emoticon_3,upvote,downvote,if_1,if_2,race,religion,gender,disability,comment,label
0,2024-01-18 08:43:57.397508+00:00,73,0,0,0,0,1,0,10,NaN,NaN,NaN,False,She might be a bright spot for a party keou on...,2
1,2024-03-24 21:43:11.490017+00:00,39,0,0,0,6,0,0,4,NaN,NaN,NaN,False,"Under Alaska law, a non-tribal member is not b...",0
2,2024-04-24 20:32:17.014931+00:00,31,0,1,1,0,0,0,10,NaN,NaN,NaN,False,in the future please spare me your strawman dr...,2
3,2023-05-28 22:00:14.214527+00:00,39,0,0,0,5,0,0,10,NaN,NaN,NaN,False,"PS: That should have been ""rot"" instead of ""co...",2
4,2023-09-09 23:12:05.689498+00:00,39,0,0,0,0,0,0,10,NaN,NaN,NaN,False,"Today, the confederate flag...tomorrow, the na...",2



Test Head:



,created_date,post_id,emoticon_1,emoticon_2,emoticon_3,upvote,downvote,if_1,if_2,race,religion,gender,disability,comment
0,2024-02-08 13:13:27.998156+00:00,72,2,0,0,4,1,0,10,NaN,NaN,NaN,False,Canada is being run by someone with the mental...
1,2024-03-01 23:33:25.547123+00:00,123,0,0,0,0,0,0,10,NaN,NaN,NaN,False,And your comment is left-wing drivel
2,2024-02-09 21:52:48.426303+00:00,120,0,0,0,3,0,0,4,NaN,NaN,NaN,False,http://talkingpointsmemo..com/dc/special-couns...
3,2024-02-17 03:43:02.980294+00:00,123,0,0,0,0,0,0,4,NaN,NaN,NaN,False,"Trump jl Blames: The Secret Service, James Com..."
4,2024-04-24 02:27:57.145155+00:00,123,0,0,0,0,0,0,11,NaN,NaN,NaN,False,It was hard enough to get the stench out of th...



Data Types:



created_date    object
post_id          int64
emoticon_1       int64
emoticon_2       int64
emoticon_3       int64
upvote           int64
downvote         int64
if_1             int64
if_2             int64
race            object
religion        object
gender          object
disability        bool
comment         object
label            int64
dtype: object


Basic Statistics:



,created_date,post_id,emoticon_1,emoticon_2,emoticon_3,upvote,downvote,if_1,if_2,race,religion,gender,disability,comment,label
count,198000,198000.000000,198000.000000,198000.000000,198000.000000,198000.000000,198000.000000,198000.000000,198000.000000,52577,52577,52577,198000,197999,198000.000000
unique,197996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,8,5,2,197842,NaN
top,2022-05-06 20:47:06.726636+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none,none,none,False,Exactly..,NaN
freq,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39682,38249,36161,195257,8,NaN
mean,NaN,68.447429,0.279768,0.048338,0.121071,2.607975,0.666394,1.906152,7.956212,NaN,NaN,NaN,NaN,NaN,0.793965
std,NaN,27.948390,1.023234,0.258477,0.481013,5.054763,2.044335,25.635752,14.839464,NaN,NaN,NaN,NaN,NaN,0.979808
min,NaN,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,39.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,NaN,NaN,NaN,NaN,NaN,0.000000
50%,NaN,72.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,6.000000,NaN,NaN,NaN,NaN,NaN,0.000000
75%,NaN,72.000000,0.000000,0.000000,0.000000,3.000000,1.000000,4.000000,10.000000,NaN,NaN,NaN,NaN,NaN,2.000000


In [4]:
train["label"].value_counts()

label
0    114173
2     62440
1     15918
3      5469
Name: count, dtype: int64

In [11]:
train.isnull().sum()

created_date         0
post_id              0
emoticon_1           0
emoticon_2           0
emoticon_3           0
upvote               0
downvote             0
if_1                 0
if_2                 0
race            145423
religion        145423
gender               0
disability           0
comment              1
label                0
dtype: int64

In [10]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
train[['gender']] = imputer.fit_transform(train[['gender']])


## 3. Exploratory Data Analysis

### 3.1 Dataset Overview

In [ ]:
print("Dataset Info:")
print(train.info())

# identify numerical and categorical columns
num_cols = train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = train.select_dtypes(include=["object", "bool"]).columns

print("Numerical Columns:", num_cols.tolist())
print("Categorical Columns:", cat_cols.tolist())

# bar chart showing count of numerical vs categorical features
feature_types = pd.Series({
    "Numerical": len(num_cols),
    "Categorical": len(cat_cols)
})

plt.figure(figsize=(5, 3))
feature_types.plot(kind="bar", color=["steelblue", "orange"], edgecolor="black")
plt.title("Feature Type Distribution")
plt.ylabel("Count")
plt.xlabel("Feature Type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Numerical histogram grid

train[num_cols].hist(figsize=(12,10), bins=40)
plt.suptitle("Numerical Feature Distributions")
plt.show()

**Observation:** 9 numerical columns and 6 categorical columns. 
upvote, downvote, emoticon_1, emoticon_2, emoticon_3 are right-skewed with most values near 0. 
if_1 is concentrated at a single value. if_2 shows a wider spread.

### 3.2 Target Distribution

In [ ]:
target_counts = train[TARGET_COL].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(target_counts.index.astype(str), target_counts.values,
            color='steelblue', edgecolor='black')
axes[0].set_title('Class Distribution - Count')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontsize=10)

axes[1].pie(target_counts.values,
            labels=target_counts.index.astype(str),
            autopct='%1.1f%%',
            startangle=140)
axes[1].set_title('Class Distribution - Share')

plt.suptitle('Target Variable Distribution', fontsize=14)
plt.tight_layout()
plt.show()

print('Imbalance ratio (max/min):', round(target_counts.max() / target_counts.min(), 2))

**Observation:** Label 0: 114,173. Label 2: 62,440. Label 1: 15,918. Label 3: 5,469. Imbalance ratio is 20.88.

### 3.3 Missing Values

In [ ]:
missing = train.isnull().sum()
missing_pct = (missing / len(train)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

plt.figure(figsize=(5, 3))
plt.bar(missing_df.index, missing_df['Missing %'], color='steelblue', edgecolor='black')
plt.title('Missing Values by Column')
plt.ylabel('Missing %')
plt.ylim(0, 100)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print(missing_df)

**Observation:** race, religion, gender each have 73.4% missing values. comment has 1 missing row (0.0005%).

### 3.4 Upvote and Downvote Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(train['upvote'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Upvote Distribution')
axes[0].set_xlabel('Upvotes')
axes[0].set_ylabel('Count')

axes[1].hist(train['downvote'], bins=40, color='orange', edgecolor='white')
axes[1].set_title('Downvote Distribution')
axes[1].set_xlabel('Downvotes')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

#KDE
sns.kdeplot(train["upvote"], label="Upvote")
sns.kdeplot(train["downvote"], label="Downvote")
plt.legend()
plt.title("KDE Distribution of Votes")
plt.show()

**Observation:** Both upvote and downvote are right-skewed. Most comments receive 0 to 5 reactions.

### 3.5 Numeric Features vs Target

In [ ]:
num_cols_box = ['upvote', 'downvote', 'emoticon_1',
                'emoticon_2', 'emoticon_3', 'if_1', 'if_2']

# mean value of each feature grouped by label
mean_df = train.groupby(TARGET_COL)[num_cols_box].mean()

plt.figure(figsize=(10, 4))
sns.heatmap(mean_df, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Mean Numeric Feature Values per Label')
plt.xlabel('Feature')
plt.ylabel('Label')
plt.tight_layout()
plt.show()

**Observation:** upvote and downvote look similar across all labels. 
emoticon_1 is slightly higher in label 0 and 2. 
if_1 is almost the same for all labels.

### 3.6 Correlation Heatmap

In [ ]:
num_cols_corr = ['upvote', 'downvote', 'emoticon_1', 'emoticon_2', 'emoticon_3', 'if_1', 'if_2']

corr = train[num_cols_corr].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap - Numeric Features')
plt.tight_layout()
plt.show()

**Observation:** No pair has correlation above 0.85. Features are largely independent of each other.

### 3.7 Outlier Detection

In [ ]:
outlier_summary = []
check_cols = ['upvote', 'downvote', 'emoticon_1', 'emoticon_2', 'emoticon_3', 'if_1', 'if_2']

for col in check_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 + 1.5 * IQR)).sum()
    outlier_summary.append({
        'Feature': col,
        'Outliers': n_out,
        'Outlier %': round(n_out / len(train) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier %', ascending=False)
print(outlier_df.to_string(index=False))

**Observation:** emoticon_1 has the highest outlier rate at 14.61%. if_1 has the lowest at 0.04%.

### 3.8 Text Feature Analysis

In [ ]:
if "comment" in train.columns:

    # fill missing values before string operations
    train["comment"] = train["comment"].fillna("").astype(str)
    test["comment"]  = test["comment"].fillna("").astype(str)

    # create text length features
    train["text_length"]     = train["comment"].apply(len)
    train["word_count"]      = train["comment"].apply(lambda x: len(x.split()))
    train["avg_word_length"] = train["text_length"] / (train["word_count"] + 1)

    test["text_length"]      = test["comment"].apply(len)
    test["word_count"]       = test["comment"].apply(lambda x: len(x.split()))
    test["avg_word_length"]  = test["text_length"] / (test["word_count"] + 1)

    # distribution of comment length
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(train["text_length"], bins=50, edgecolor="white")
    axes[0].set_title("Comment Length (Characters)")
    axes[0].set_xlabel("Char Count")

    axes[1].hist(train["word_count"], bins=50, edgecolor="white")
    axes[1].set_title("Comment Length (Words)")
    axes[1].set_xlabel("Word Count")

    plt.tight_layout()
    plt.show()

    # average text length and word count per label
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    train.groupby("label")["text_length"].mean().plot(
        kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
    axes[0].set_title("Average Text Length by Label")
    axes[0].set_ylabel("Average Characters")
    axes[0].set_xlabel("Label")

    train.groupby("label")["word_count"].mean().plot(
        kind="bar", ax=axes[1], color="orange", edgecolor="black")
    axes[1].set_title("Average Word Count by Label")
    axes[1].set_ylabel("Average Words")
    axes[1].set_xlabel("Label")

    plt.tight_layout()
    plt.show()

    # KDE shows text length distribution shape per label
    plt.figure(figsize=(8, 5))
    for lbl in sorted(train["label"].unique()):
        sns.kdeplot(
            train[train["label"] == lbl]["text_length"],
            label=f"Label {lbl}"
        )
    plt.title("Text Length Distribution by Label")
    plt.xlabel("Text Length")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

**Observation:** Most comments are short in length. 
Label 3 comments are the shortest on average. 
Label 1 comments are the longest on average.

### 3.9 Temporal Analysis

In [ ]:
train['created_date'] = pd.to_datetime(train['created_date'], errors='coerce')
train['hour']    = train['created_date'].dt.hour
train['month']   = train['created_date'].dt.month
train['weekday'] = train['created_date'].dt.dayofweek

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

train['month'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0][0], color='steelblue', edgecolor='black')
axes[0][0].set_title('Posts per Month')
axes[0][0].set_xlabel('Month')
axes[0][0].set_ylabel('Count')

train['weekday'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0][1], color='orange', edgecolor='black')
axes[0][1].set_title('Posts per Weekday')
axes[0][1].set_xlabel('Day (0=Mon, 6=Sun)')
axes[0][1].set_ylabel('Count')

axes[1][0].hist(train['hour'], bins=24, color='steelblue', edgecolor='white')
axes[1][0].set_title('Posting Hour Distribution')
axes[1][0].set_xlabel('Hour')
axes[1][0].set_ylabel('Count')

hour_label = pd.crosstab(train['hour'], train['label'])
sns.heatmap(hour_label, cmap='Blues', annot=False, ax=axes[1][1])
axes[1][1].set_title('Hour vs Label Heatmap')
axes[1][1].set_xlabel('Label')
axes[1][1].set_ylabel('Hour')

plt.tight_layout()
plt.show()


**Observation:** Comments are posted across all months and hours fairly evenly. 
No particular time or day stands out. 
Posting patterns do not vary much by label.

### 3.10 Emoticon Usage Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train[['emoticon_1','emoticon_2','emoticon_3']].sum().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Total Emoticon Usage')
axes[0].set_xlabel('Emoticon Group')
axes[0].set_ylabel('Count')

emoticon_sum = train.groupby('label')[['emoticon_1','emoticon_2','emoticon_3']].sum()
emoticon_sum.plot(kind='bar', stacked=True, ax=axes[1], edgecolor='black')
axes[1].set_title('Emoticon Usage by Label')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

**Observation:** emoticon_1 is the most used across all labels. 
emoticon_2 has the lowest overall usage. Label 0 has the highest total emoticon count 
due to its larger sample size.

### 3.11 Engagement Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(train['upvote'], train['downvote'], alpha=0.3, color='steelblue')
axes[0].set_title('Upvote vs Downvote')
axes[0].set_xlabel('Upvote')
axes[0].set_ylabel('Downvote')

sns.boxplot(data=train, x='label', y='upvote', ax=axes[1])
axes[1].set_title('Upvotes by Label')
axes[1].set_xlabel('Label')

sns.boxplot(data=train, x='label', y='downvote', ax=axes[2])
axes[2].set_title('Downvotes by Label')
axes[2].set_xlabel('Label')

plt.tight_layout()
plt.show()

**Observation:** Most comments cluster near 0 upvotes and 0 downvotes. 
Label 3 has the lowest median upvotes and downvotes. 
All labels have many high-vote outliers.

### 3.12 Category Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

train.groupby('label')['upvote'].mean().plot(
    kind='bar', ax=axes[0][0], color='steelblue', edgecolor='black')
axes[0][0].set_title('Average Upvotes per Label')
axes[0][0].set_xlabel('Label')

train.groupby('label')['text_length'].mean().plot(
    kind='bar', ax=axes[0][1], color='orange', edgecolor='black')
axes[0][1].set_title('Average Text Length per Label')
axes[0][1].set_xlabel('Label')

train.groupby('label')[['emoticon_1','emoticon_2','emoticon_3']].mean().plot(
    kind='bar', ax=axes[1][0], edgecolor='black')
axes[1][0].set_title('Average Emoticon Usage per Label')
axes[1][0].set_xlabel('Label')

engagement = train.groupby('label')[['upvote','downvote']].mean()
sns.heatmap(engagement, annot=True, fmt='.2f', ax=axes[1][1], cmap='Blues')
axes[1][1].set_title('Engagement Heatmap by Label')

plt.tight_layout()
plt.show()


**Observation:** Label 1 has the highest average upvotes at 2.85 and longest text at 335 chars. 
Label 3 has the lowest across upvotes (2.04), text length (194 chars) and emoticon usage. 
emoticon_1 is consistently higher than emoticon_2 and emoticon_3 across all labels.

### 3.13 WordCloud

In [ ]:
from wordcloud import WordCloud

text_all = ' '.join(train['comment'].astype(str))
wc = WordCloud(width=800, height=400, background_color='white').generate(text_all)
plt.figure(figsize=(10, 4))
plt.imshow(wc)
plt.axis('off')
plt.title('WordCloud - All Comments')
plt.show()

for label in sorted(train['label'].unique()):
    text = ' '.join(train[train['label'] == label]['comment'].astype(str))
    wc = WordCloud(width=800, height=300, background_color='white').generate(text)
    plt.figure(figsize=(10, 3))
    plt.imshow(wc)
    plt.axis('off')
    plt.title(f'WordCloud - Label {label}')
    plt.show()

**Observation:** Common words like people, would and one appear in all labels. 
Each label has some words that appear more than others. 
This shows each label has a slightly different topic pattern.

 ### 3.14 Feature Relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(train['text_length'], train['upvote'], alpha=0.3, color='steelblue')
axes[0].set_title('Text Length vs Upvotes')
axes[0].set_xlabel('Text Length')
axes[0].set_ylabel('Upvotes')

axes[1].scatter(train['word_count'], train['upvote'], alpha=0.3, color='orange')
axes[1].set_title('Word Count vs Upvotes')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Upvotes')

axes[2].scatter(train['emoticon_1'], train['upvote'], alpha=0.3, color='green')
axes[2].set_title('Emoticon 1 vs Upvotes')
axes[2].set_xlabel('Emoticon 1')
axes[2].set_ylabel('Upvotes')

plt.tight_layout()
plt.show()


**Observation:** No strong linear relationship between text length, word count 
or emoticon usage and upvotes. Engagement signals are independent of comment length.

### 3.15 Statistical Summary

In [ ]:
print('Mean Upvotes per Label:')
print(train.groupby('label')['upvote'].mean().round(2))

print('\nMean Text Length per Label:')
print(train.groupby('label')['text_length'].mean().round(2))

print('\nMean Emoticon Usage per Label:')
print(train.groupby('label')[['emoticon_1','emoticon_2','emoticon_3']].mean().round(3))

**Observation:** Label 3 consistently has the lowest mean values across upvotes (2.04), 
text length (194 chars) and all emoticon groups. 
Label 1 has the highest mean upvotes (2.85) and longest mean text length (335 chars).

## 4. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['comment'] = df['comment'].astype(str).fillna('')

    # text length features
    df['text_length']     = df['comment'].apply(len)
    df['word_count']      = df['comment'].apply(lambda x: len(x.split()))
    df['unique_words']    = df['comment'].apply(lambda x: len(set(x.lower().split())))
    df['avg_word_length'] = df['comment'].apply(
        lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
    df['lexical_density'] = df['unique_words'] / (df['word_count'] + 1)
    df['char_per_word']   = df['text_length'] / (df['word_count'] + 1)

    # punctuation and style features
    df['exclaim_count']  = df['comment'].apply(lambda x: x.count('!'))
    df['question_count'] = df['comment'].apply(lambda x: x.count('?'))
    df['caps_count']     = df['comment'].apply(lambda x: sum(1 for ch in x if ch.isupper()))
    df['caps_ratio']     = df['caps_count'] / (df['text_length'] + 1)
    df['digit_count']    = df['comment'].apply(lambda x: sum(1 for ch in x if ch.isdigit()))
    df['url_count']      = df['comment'].apply(
        lambda x: len(re.findall(r'http\S+|www\.\S+', x)))
    df['mention_count']  = df['comment'].apply(
        lambda x: len(re.findall(r'@\w+', x)))

    # date-based features
    if 'created_date' in df.columns:
        df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
        df['hour']         = df['created_date'].dt.hour
        df['day_of_week']  = df['created_date'].dt.dayofweek
        df['day_of_month'] = df['created_date'].dt.day
        df['month']        = df['created_date'].dt.month
        df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
        df['is_night']     = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)

    # vote interaction features
    df['vote_total']       = df['upvote'] + df['downvote']
    df['vote_net']         = df['upvote'] - df['downvote']
    df['vote_ratio']       = df['upvote'] / (df['vote_total'] + 1)
    df['log_upvote']       = np.log1p(df['upvote'])
    df['log_downvote']     = np.log1p(df['downvote'])
    df['is_controversial'] = ((df['upvote'] > 0) & (df['downvote'] > 0)).astype(int)

    # emoticon group features
    df['emoticon_total'] = df['emoticon_1'] + df['emoticon_2'] + df['emoticon_3']
    df['emoticon_any']   = (df['emoticon_total'] > 0).astype(int)

    # sensitive topic flags from system detection columns
    for col in ['race', 'religion', 'gender']:
        if col in df.columns:
            df[col + '_flag'] = df[col].notna().astype(int)
    flag_cols = [col + '_flag' for col in ['race', 'religion', 'gender'] if col + '_flag' in df.columns]
    df['sensitive_total'] = df[flag_cols].sum(axis=1)
    df['sensitive_any']   = (df['sensitive_total'] > 0).astype(int)

    return df

train = engineer_features(train)
test  = engineer_features(test)

print('Train shape after feature engineering:', train.shape)
print('Test  shape after feature engineering:', test.shape)

## 5. Preprocessing

### 5.1 Encode Categorical Features

In [ ]:
# fill missing with 'unknown' before encoding
for col in ['race', 'religion', 'gender']:
    train[col] = train[col].fillna('unknown')
    test[col]  = test[col].fillna('unknown')

# label encode each categorical column
# fit on combined train+test to handle unseen labels in test
le_dict = {}
for col in ['race', 'religion', 'gender']:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0)
    le.fit(combined)
    train[col + '_enc'] = le.transform(train[col])
    test[col + '_enc']  = le.transform(test[col])
    le_dict[col] = le

# disability is boolean, convert to int directly
train['disability'] = train['disability'].astype(int)
test['disability']  = test['disability'].astype(int)

print('Encoding done.')
print(train[['race_enc', 'religion_enc', 'gender_enc', 'disability']].head())

### 5.2 Select Tabular Features

In [ ]:
num_cols = [
    'upvote', 'downvote', 'if_1', 'if_2',
    'text_length', 'word_count', 'avg_word_length',
    'unique_words', 'lexical_density',
    'caps_ratio', 'digit_count',
    'hour', 'day_of_week', 'month',
    'is_weekend', 'is_night',
    'vote_total', 'vote_ratio',
    'log_upvote', 'log_downvote',
    'emoticon_total', 'sensitive_total'
]

num_cols = [col for col in num_cols if col in train.columns]
print('Total tabular features:', len(num_cols))

### 5.3 Scale Tabular Features

In [ ]:
# scaling for Logistic Regression only
scaler = StandardScaler()
X_train_tab_scaled = scaler.fit_transform(train[num_cols])
X_test_tab_scaled  = scaler.transform(test[num_cols])

X_train_tab = train[num_cols].values
X_test_tab  = test[num_cols].values

print('Scaled shape  :', X_train_tab_scaled.shape)
print('Unscaled shape:', X_train_tab.shape)

## 6. TF-IDF Text Vectorization

### 6.1 Word-level TF-IDF

In [ ]:
train['comment'] = train['comment'].fillna('').astype(str)
test['comment']  = test['comment'].fillna('').astype(str)

tfidf_word = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1, 2),
    min_df=3,
    stop_words='english'
)

X_train_word = tfidf_word.fit_transform(train['comment'])
X_test_word  = tfidf_word.transform(test['comment'])

print('Word TF-IDF shape:', X_train_word.shape)

### 6.2 Character-level TF-IDF

In [ ]:
tfidf_char = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    max_features=5000
)

X_train_char = tfidf_char.fit_transform(train['comment'])
X_test_char  = tfidf_char.transform(test['comment'])

print('Char TF-IDF shape:', X_train_char.shape)

## 7. Feature Combination

In [ ]:
X_train_full_scaled = hstack([X_train_word, X_train_char, csr_matrix(X_train_tab_scaled)])
X_test_full_scaled  = hstack([X_test_word,  X_test_char,  csr_matrix(X_test_tab_scaled)])

X_train_full = hstack([X_train_word, X_train_char, csr_matrix(X_train_tab)])
X_test_full  = hstack([X_test_word,  X_test_char,  csr_matrix(X_test_tab)])

print('Scaled train shape  :', X_train_full_scaled.shape)
print('Unscaled train shape:', X_train_full.shape)

## 8. Train / Validation Split

In [ ]:
y = train[TARGET_COL]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y,
    test_size=0.1,
    stratify=y,
    random_state=RANDOM_STATE
)

X_tr_scaled, X_val_scaled, _, _ = train_test_split(
    X_train_full_scaled, y,
    test_size=0.1,
    stratify=y,
    random_state=RANDOM_STATE
)

print('Train size    :', X_tr.shape)
print('Validation size:', X_val.shape)

## 9. Model Training

### 9.1 Logistic Regression with Hyperparameter Tuning

In [ ]:
X_sample_scaled, _, y_sample, _ = train_test_split(
    X_tr_scaled, y_tr,
    train_size=0.2,
    stratify=y_tr,
    random_state=RANDOM_STATE
)

param_dist = {
    'C': [0.1, 0.5, 1.0, 2.0, 5.0],
    'max_iter': [200, 500]
}

lr_search = RandomizedSearchCV(
    LogisticRegression(solver='saga', n_jobs=-1),
    param_dist,
    n_iter=5,
    cv=3,
    scoring='f1_macro',
    random_state=RANDOM_STATE
)

lr_search.fit(X_sample_scaled, y_sample)
print('Best LR params:', lr_search.best_params_)
print('Best CV F1 Macro:', round(lr_search.best_score_, 4))

In [ ]:
lr_model = LogisticRegression(
    solver='saga',
    n_jobs=-1,
    **lr_search.best_params_
)

lr_model.fit(X_tr_scaled, y_tr)
pred_lr = lr_model.predict(X_val_scaled)

print('Logistic Regression')
print('Accuracy  :', round(accuracy_score(y_val, pred_lr), 4))
print('F1 Macro  :', round(f1_score(y_val, pred_lr, average='macro'), 4))
print('F1 Weighted:', round(f1_score(y_val, pred_lr, average='weighted'), 4))
print()
print(classification_report(y_val, pred_lr))

### 9.2 XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(X_tr, y_tr)
pred_xgb = xgb_model.predict(X_val)

print('XGBoost')
print('Accuracy  :', round(accuracy_score(y_val, pred_xgb), 4))
print('F1 Macro  :', round(f1_score(y_val, pred_xgb, average='macro'), 4))
print('F1 Weighted:', round(f1_score(y_val, pred_xgb, average='weighted'), 4))
print()
print(classification_report(y_val, pred_xgb))

### 9.3 LightGBM

In [ ]:
counter = Counter(y_tr)
total = sum(counter.values())
class_weights = {
    cls: total / (len(counter) * count)
    for cls, count in counter.items()
}

print('Class distribution:', dict(counter))
print('Class weights:', class_weights)

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight=class_weights,
    random_state=RANDOM_STATE,
    force_col_wise=True,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(X_tr, y_tr)
pred_lgb = lgb_model.predict(X_val)

print('LightGBM')
print('Accuracy  :', round(accuracy_score(y_val, pred_lgb), 4))
print('F1 Macro  :', round(f1_score(y_val, pred_lgb, average='macro'), 4))
print('F1 Weighted:', round(f1_score(y_val, pred_lgb, average='weighted'), 4))
print()
print(classification_report(y_val, pred_lgb))

### 9.4 Pipeline Setup

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ── ColumnTransformer for Logistic Regression (needs scaling) ───────────────
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('tfidf_word', TfidfVectorizer(
            max_features=12000, ngram_range=(1, 2),
            min_df=3, stop_words='english'
        ), 'comment'),
        ('tfidf_char', TfidfVectorizer(
            analyzer='char', ngram_range=(3, 5), max_features=5000
        ), 'comment'),
        ('scaler', StandardScaler(), num_cols)
    ],
    remainder='drop'
)

# ── ColumnTransformer for LightGBM (no scaling needed) ─────────────────────
preprocessor_lgb = ColumnTransformer(
    transformers=[
        ('tfidf_word', TfidfVectorizer(
            max_features=12000, ngram_range=(1, 2),
            min_df=3, stop_words='english'
        ), 'comment'),
        ('tfidf_char', TfidfVectorizer(
            analyzer='char', ngram_range=(3, 5), max_features=5000
        ), 'comment'),
        ('passthrough', 'passthrough', num_cols)
    ],
    remainder='drop'
)

# ── Pipeline 1: Logistic Regression ────────────────────────────────────────
pipe_lr = Pipeline([
    ('preprocessor', preprocessor_lr),
    ('model', LogisticRegression(
        solver='saga', C=2.0, max_iter=500, n_jobs=-1
    ))
])

# ── Pipeline 2: LightGBM ────────────────────────────────────────────────────
pipe_lgb = Pipeline([
    ('preprocessor', preprocessor_lgb),
    ('model', LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight=class_weights,  # already defined in Section 9.3
        random_state=RANDOM_STATE,
        force_col_wise=True,
        n_jobs=-1,
        verbose=-1
    ))
])

print('Pipelines defined successfully.')
print('pipe_lr  steps:', [s[0] for s in pipe_lr.steps])
print('pipe_lgb steps:', [s[0] for s in pipe_lgb.steps])


### 9.5 Train & Evaluate Pipelines


In [ ]:
# Pipeline needs raw text + numeric cols in a single DataFrame
input_cols   = ['comment'] + num_cols
X_pipe_tr    = train[input_cols]
X_pipe_test  = test[input_cols]
y_pipe       = train[TARGET_COL]

# Train / validation split
X_pipe_tr, X_pipe_val, y_pipe_tr, y_pipe_val = train_test_split(
    X_pipe_tr, y_pipe,
    test_size=0.1,
    stratify=y_pipe,
    random_state=RANDOM_STATE
)

print('Pipeline train size :', X_pipe_tr.shape)
print('Pipeline val size   :', X_pipe_val.shape)

# ── Fit Logistic Regression Pipeline ───────────────────────────────────────
print('\nFitting Logistic Regression Pipeline...')
pipe_lr.fit(X_pipe_tr, y_pipe_tr)
pred_pipe_lr = pipe_lr.predict(X_pipe_val)

print('Logistic Regression Pipeline')
print('Accuracy :', round(accuracy_score(y_pipe_val, pred_pipe_lr), 4))
print('F1 Macro :', round(f1_score(y_pipe_val, pred_pipe_lr, average='macro'), 4))
print(classification_report(y_pipe_val, pred_pipe_lr))

# ── Fit LightGBM Pipeline ───────────────────────────────────────────────────
print('\nFitting LightGBM Pipeline...')
pipe_lgb.fit(X_pipe_tr, y_pipe_tr)
pred_pipe_lgb = pipe_lgb.predict(X_pipe_val)

print('LightGBM Pipeline')
print('Accuracy :', round(accuracy_score(y_pipe_val, pred_pipe_lgb), 4))
print('F1 Macro :', round(f1_score(y_pipe_val, pred_pipe_lgb, average='macro'), 4))
print(classification_report(y_pipe_val, pred_pipe_lgb))


## 10. Model Comparison

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost', 'LightGBM'],
    'Accuracy': [
        round(accuracy_score(y_val, pred_lr), 4),
        round(accuracy_score(y_val, pred_xgb), 4),
        round(accuracy_score(y_val, pred_lgb), 4)
    ],
    'F1 Macro': [
        round(f1_score(y_val, pred_lr, average='macro'), 4),
        round(f1_score(y_val, pred_xgb, average='macro'), 4),
        round(f1_score(y_val, pred_lgb, average='macro'), 4)
    ],
    'F1 Weighted': [
        round(f1_score(y_val, pred_lr, average='weighted'), 4),
        round(f1_score(y_val, pred_xgb, average='weighted'), 4),
        round(f1_score(y_val, pred_lgb, average='weighted'), 4)
    ]
}).sort_values('F1 Macro', ascending=False)

print(results_df.to_string(index=False))

In [ ]:
# confusion matrix 
model_preds = [
    ('Logistic Regression', pred_lr),
    ('XGBoost',             pred_xgb),
    ('LightGBM',            pred_lgb)
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices', fontsize=16, fontweight='bold', y=1.02)

for ax, (name, pred) in zip(axes, model_preds):
    cm = confusion_matrix(y_val, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                linewidths=0.5, ax=ax)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('Actual Label')

plt.tight_layout()
plt.show()

## 11. Final Submission

In [ ]:
y_full = train[TARGET_COL]

counter_full = Counter(y_full)
total_full = sum(counter_full.values())
class_weights_full = {
    cls: total_full / (len(counter_full) * count)
    for cls, count in counter_full.items()
}

lgb_final = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight=class_weights_full,
    random_state=RANDOM_STATE,
    force_col_wise=True,
    n_jobs=-1,
    verbose=-1
)

lgb_final.fit(X_train_full, y_full)

pred_test = lgb_final.predict(X_test_full)

submission = sample_sub.copy()
submission['label'] = pred_test
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('Submission saved.')
print('Label distribution:')
print(submission['label'].value_counts().sort_index())